# 🧠 Bilişsel Performans Skoru Tahmini
## Google Yapay Zeka ve Teknoloji Akademisi — Datathon 2025

Bu notebook'ta bireylerin uyku düzeni, yaşam alışkanlıkları ve günlük durumlarına
ait değişkenler kullanılarak **bilişsel performans skoru** tahmin edilmektedir.

### Yaklaşımımız
1. Keşifsel Veri Analizi (EDA)
2. Veri Temizleme
3. Feature Engineering
4. LightGBM + XGBoost + CatBoost Ensemble
5. 5-Fold Cross Validation

## 1. Kütüphaneler

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

print("Kütüphaneler yüklendi ✓")

## 2. Veri Yükleme

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')

print(f"Train seti: {train.shape[0]} satır, {train.shape[1]} sütun")
print(f"Test  seti: {test.shape[0]} satır, {test.shape[1]} sütun")
train.head()

## 3. Keşifsel Veri Analizi (EDA)
Veriyi anlamak için önce genel istatistiklere, sonra görsel analizlere bakıyoruz.

In [ ]:
print("=== HEDEF DEĞİŞKEN İSTATİSTİKLERİ ===")
print(train['bilissel_performans_skoru'].describe().round(2))

print("\n=== EKSİK DEĞERLER ===")
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hedef dağılımı
axes[0].hist(train['bilissel_performans_skoru'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Hedef Değişken Dağılımı')
axes[0].set_xlabel('Bilişsel Performans Skoru')
axes[0].set_ylabel('Frekans')

# Korelasyon
num_cols = ['stres_skoru','rem_yuzdesi','gunluk_calisma_saati','derin_uyku_yuzdesi',
            'gecelik_uyanma_sayisi','uykuya_dalma_suresi_dk','gunluk_adim_sayisi',
            'dinlenik_nabiz_bpm','yas','uyku_oncesi_kafein_mg']
corr = train[num_cols].corrwith(train['bilissel_performans_skoru']).sort_values()
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
axes[1].barh(corr.index, corr.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Özellikler ile Hedef Korelasyonu')
axes[1].set_xlabel('Korelasyon Katsayısı')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ruh sağlığı
ruh_means = train.groupby('ruh_sagligi_durumu')['bilissel_performans_skoru'].mean().sort_values()
axes[0].barh(ruh_means.index, ruh_means.values, color=['#e74c3c','#e67e22','#3498db','#2ecc71'])
axes[0].set_title('Ruh Sağlığı Durumuna Göre Ortalama Skor')
axes[0].set_xlabel('Ortalama Skor')

# Meslek
meslek_means = train.groupby('meslek')['bilissel_performans_skoru'].mean().sort_values()
axes[1].barh(meslek_means.index, meslek_means.values, color='steelblue')
axes[1].set_title('Mesleğe Göre Ortalama Skor')
axes[1].set_xlabel('Ortalama Skor')

plt.tight_layout()
plt.show()

print("Bulgular:")
print("→ Ruh sağlığı durumu skoru güçlü şekilde etkiliyor")
print("→ Emekliler en yüksek, Avukatlar en düşük skora sahip")

## 4. Veri Temizleme
EDA'dan öğrendiklerimizi uyguluyoruz: kategorik tutarsızlıklar ve aşırı uç değerler temizleniyor.

In [ ]:
MAPPING = {
    'South Korea': 'Guney Kore',
    'Spain':       'Ispanya',
    'Sweden':      'Isvec',
    'Lawyer':      'Avukat'
}
train = train.replace(MAPPING)
test  = test.replace(MAPPING)

# Outlier temizliği kaldırıldı
print(f"Toplam satır sayısı: {len(train)}")

## 5. Feature Engineering
Mevcut sütunları birleştirerek modele yardımcı olacak 10 yeni anlamlı özellik türetiyoruz.

In [ ]:
def engineer(df):
    df = df.copy()

    # ── UYKU KALİTESİ ────────────────────────────────
    df['kaliteli_uyku']        = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['uyku_verimi']          = df['kaliteli_uyku'] / (df['gecelik_uyanma_sayisi'] + 1)
    df['dalma_uyanma']         = df['uykuya_dalma_suresi_dk'] * df['gecelik_uyanma_sayisi']
    df['toplam_kotu_uyku']     = df['uykuya_dalma_suresi_dk'] + (df['gecelik_uyanma_sayisi'] * 10)
    df['uyku_etkinligi']       = df['kaliteli_uyku'] / (df['uykuya_dalma_suresi_dk'] + 1)
    df['hafif_uyku']           = 100 - df['kaliteli_uyku']

    # ── STRES ETKİLEŞİMLERİ ──────────────────────────
    df['stres_calisma']        = df['stres_skoru'] * df['gunluk_calisma_saati']
    df['stres_uyanma']         = df['stres_skoru'] * df['gecelik_uyanma_sayisi']
    df['stres_rem']            = df['stres_skoru'] * (10 - df['rem_yuzdesi'] / 10)
    df['stres_uyku_verimi']    = df['stres_skoru'] * df['uyku_verimi']
    df['stres_kafein']         = df['stres_skoru'] * df['uyku_oncesi_kafein_mg'].fillna(0)
    df['stres_kare']           = df['stres_skoru'] ** 2

    # ── AKTİF YAŞAM ──────────────────────────────────
    df['aktivite_skoru']       = df['gunluk_adim_sayisi'] / (df['gunluk_calisma_saati'] + 1)
    df['nabiz_aktivite']       = df['dinlenik_nabiz_bpm'] / (df['gunluk_adim_sayisi'] / 1000 + 1)
    df['adim_kare']            = df['gunluk_adim_sayisi'] ** 2 / 1e8

    # ── KÖTÜ GECE ALIŞ. ──────────────────────────────
    df['gece_uyari']           = df['uyku_oncesi_kafein_mg'].fillna(0) + df['uyku_oncesi_ekran_suresi_dk']
    df['kafein_ekran']         = df['uyku_oncesi_kafein_mg'].fillna(0) * df['uyku_oncesi_ekran_suresi_dk']

    # ── YAŞ ETKİLEŞİMLERİ ────────────────────────────
    df['yas_stres']            = df['yas'] * df['stres_skoru']
    df['yas_uyku']             = df['yas'] * df['uyku_verimi']
    df['yas_aktivite']         = df['yas'] * df['gunluk_adim_sayisi'] / 1000

    # ── VKİ ETKİLEŞİMLERİ ────────────────────────────
    vki = df['vucut_kitle_indeksi'].fillna(df['vucut_kitle_indeksi'].median())
    df['vki_stres']            = vki * df['stres_skoru']
    df['vki_aktivite']         = vki / (df['gunluk_adim_sayisi'] / 1000 + 1)

    # ── ŞEKERLEME ─────────────────────────────────────
    df['sekerleme_fayda']      = df['sekerleme_suresi_dk'] / (df['stres_skoru'] + 1)

    # ── HAFTA SONU ────────────────────────────────────
    df['hafta_sonu_flag']      = (df['gun_tipi'] == 'Hafta sonu').astype(int)
    df['hafta_sonu_uyku']      = df['hafta_sonu_flag'] * df['kaliteli_uyku']
    df['hafta_sonu_adim']      = df['hafta_sonu_flag'] * df['gunluk_adim_sayisi']

    # ── SICAKLIK ──────────────────────────────────────
    df['sicaklik_uyku']        = df['oda_sicakligi_celsius'] * df['uyku_verimi']
    df['ideal_sicaklik']       = ((df['oda_sicakligi_celsius'] - 18) ** 2)

    # ── RUH SAĞLIĞI ENCODING ─────────────────────────
    ruh_map = {
        'Saglikli':               3,
        'Anksiyete':              2,
        'Depresyon':              1,
        'Anksiyete ve depresyon': 0
    }
    df['ruh_sagligi_encoded']  = df['ruh_sagligi_durumu'].map(ruh_map).fillna(1.5)
    df['ruh_stres']            = df['ruh_sagligi_encoded'] * df['stres_skoru']
    df['ruh_uyku']             = df['ruh_sagligi_encoded'] * df['uyku_verimi']


    meslek_map = {
    'Emekli':                      7.94,
    'Ev Hanimi':                   7.06,
    'Serbest Calisan':             6.99,
    'Egitimci':                    6.34,
    'Muhendis':                    6.22,
    'Satis ve Pazarlama Calisani': 6.01,
    'Yonetici':                    5.73,
    'Ogrenci':                     5.54,
    'Lojistik Calisani':           5.12,
    'Saglik Personeli':            4.93,
    'Avukat':                      4.72
    }
    df['meslek_encoded'] = df['meslek'].map(meslek_map).fillna(5.5)

   


    # ── SOSYAL JETLAG ─────────────────────────────────
    # Hafta sonu ile hafta içi uyku farkı ne kadar büyükse performans düşer
    df['sosyal_jetlag']        = df['hafta_sonu_uyku_farki_saat'].abs()
    df['jetlag_stres']         = df['sosyal_jetlag'] * df['stres_skoru']
    df['jetlag_uyku']          = df['sosyal_jetlag'] * df['uyku_verimi']

    # ── KRONOTİP × GÜN TİPİ ──────────────────────────
    # Gece kuşu hafta içi çalışıyorsa kötü performans
    df['gece_hafta_ici']       = (
        (df['kronotip'] == 'Gece') & 
        (df['gun_tipi'] == 'Hafta ici')
    ).astype(int)

    df['sabah_hafta_sonu']     = (
        (df['kronotip'] == 'Sabah') & 
        (df['gun_tipi'] == 'Hafta sonu')
    ).astype(int)
    
    return df

train = engineer(train)
test  = engineer(test)
print(f"Toplam özellik sayısı: {len(train.columns) - 2}")

## 6. Model Eğitimi
3 farklı güçlü algoritma kullanıyoruz ve 5-Fold CV ile değerlendiriyoruz.

- **LightGBM**: Hızlı gradient boosting
- **XGBoost**: Güçlü ve yaygın gradient boosting
- **CatBoost**: Kategorik verileri doğal olarak işleyen gradient boosting

In [ ]:
TARGET   = 'bilissel_performans_skoru'
cat_cols = ['meslek','ruh_sagligi_durumu','gun_tipi']

X      = train.drop(columns=['id', TARGET])
y      = train[TARGET]
X_test = test.drop(columns=['id'])

drop_features = [
    'sabah_hafta_sonu', 'gece_hafta_ici', 'cinsiyet',
    'mevsim', 'kronotip', 'hafta_sonu_flag',
    'ulke', 'adim_kare', 'sosyal_jetlag'
]
X      = X.drop(columns=drop_features)
X_test = X_test.drop(columns=drop_features)

encoder    = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_enc      = X.copy()
X_test_enc = X_test.copy()
X_enc[cat_cols]      = encoder.fit_transform(X[cat_cols].astype(str))
X_test_enc[cat_cols] = encoder.transform(X_test[cat_cols].astype(str))

X_cat      = X.copy()
X_test_cat = X_test.copy()
for c in cat_cols:
    X_cat[c]      = X_cat[c].astype(str).fillna('Bilinmiyor')
    X_test_cat[c] = X_test_cat[c].astype(str).fillna('Bilinmiyor')
cat_idx = [list(X_cat.columns).index(c) for c in cat_cols]

print("Veri hazırlandı ✓")
print(f"Eğitim: {X.shape}, Test: {X_test.shape}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
train['target_bin'] = pd.qcut(y, q=10, labels=False, duplicates='drop')
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
lgb_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))
lgb_rmses, xgb_rmses, cat_rmses = [], [], []
for fold, (tr_idx, val_idx) in enumerate(kf.split(X, train['target_bin'])):
    # LightGBM
    lgb_m = lgb.LGBMRegressor(
        n_estimators=1692, learning_rate=0.0116, num_leaves=74,
        min_child_samples=51, subsample=0.8897, colsample_bytree=0.6177,
        reg_alpha=0.0184, reg_lambda=1.3183, random_state=42, n_jobs=-1, verbose=-1)
    lgb_m.fit(X_enc.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_enc.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
    p_lgb = np.clip(lgb_m.predict(X_enc.iloc[val_idx]), 0, 10)
    lgb_rmses.append(np.sqrt(mean_squared_error(y.iloc[val_idx], p_lgb)))
    lgb_preds += np.clip(lgb_m.predict(X_test_enc), 0, 10) / 10
    # XGBoost
    xgb_m = xgb.XGBRegressor(
        n_estimators=2710, learning_rate=0.0179, max_depth=5,
        subsample=0.6149, colsample_bytree=0.6031,
        reg_alpha=0.2961, reg_lambda=1.3764,
        random_state=42, n_jobs=-1, eval_metric='rmse', verbosity=0,
        early_stopping_rounds=100)
    xgb_m.fit(X_enc.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_enc.iloc[val_idx], y.iloc[val_idx])], verbose=False)
    p_xgb = np.clip(xgb_m.predict(X_enc.iloc[val_idx]), 0, 10)
    xgb_rmses.append(np.sqrt(mean_squared_error(y.iloc[val_idx], p_xgb)))
    xgb_preds += np.clip(xgb_m.predict(X_test_enc), 0, 10) / 10
    # CatBoost
    cat_m = CatBoostRegressor(
        iterations=1652, learning_rate=0.0308, depth=6,
        l2_leaf_reg=2.59, subsample=0.78,
        random_seed=42, verbose=0,
        early_stopping_rounds=100, eval_metric='RMSE')
    cat_m.fit(X_cat.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X_cat.iloc[val_idx], y.iloc[val_idx]),
              cat_features=cat_idx)
    p_cat = np.clip(cat_m.predict(X_cat.iloc[val_idx]), 0, 10)
    cat_rmses.append(np.sqrt(mean_squared_error(y.iloc[val_idx], p_cat)))
    cat_preds += np.clip(cat_m.predict(X_test_cat), 0, 10) / 10
    print(f"Fold {fold+1} → LGB:{lgb_rmses[-1]:.4f} | XGB:{xgb_rmses[-1]:.4f} | CAT:{cat_rmses[-1]:.4f}")
print(f"\n{'='*60}")
print(f"LightGBM     RMSE: {np.mean(lgb_rmses):.4f} ± {np.std(lgb_rmses):.4f}")
print(f"XGBoost      RMSE: {np.mean(xgb_rmses):.4f} ± {np.std(xgb_rmses):.4f}")
print(f"CatBoost     RMSE: {np.mean(cat_rmses):.4f} ± {np.std(cat_rmses):.4f}")

## 7. Sonuçlar

In [ ]:
print("="*50)
print(f"LightGBM  RMSE: {np.mean(lgb_rmses):.4f} ± {np.std(lgb_rmses):.4f}")
print(f"XGBoost   RMSE: {np.mean(xgb_rmses):.4f} ± {np.std(xgb_rmses):.4f}")
print(f"CatBoost  RMSE: {np.mean(cat_rmses):.4f} ± {np.std(cat_rmses):.4f}")
print("="*50)

fi = pd.Series(lgb_m.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
plt.barh(fi.index[::-1], fi.values[::-1], color='steelblue')
plt.title('En Önemli 15 Özellik (LightGBM)')
plt.xlabel('Önem Skoru')
plt.tight_layout()
plt.show()


## 8. Submission

In [ ]:
# RF çıkarıldı, sadece iyi 3 model
ensemble = 0.25 * lgb_preds + 0.25 * xgb_preds + 0.50 * cat_preds
submission = pd.DataFrame({
    'id': test['id'],
    'bilissel_performans_skoru': ensemble
})

submission.to_csv('submission.csv', index=False)
print("Submission kaydedildi ✓")
print(submission.head(10))